# Cálculo de sazonalidade com NDVI

-- Calculo importante para difirenciar tipos indicies de NDVI de acordo com o tempo e ciclos de um ano.

***Com 2 ciclos: A agricultura intensiva (soja seguida de milho/sorgo) gera um sinal fortíssimo no 2º harmônico (magnitude_2, sin_2, cos_2) porque há dois plantios, dois picos de biomassa e duas colheitas/dessecações no mesmo ano. A pastagem quase não tem energia no 2º ciclo (ela segue o regime natural de chuvas), o que cria uma fronteira clara para o classificador separar lavoura de pasto.***

***O ponto fundamental é que o algoritmo não calcula simplesmente a média do NDVI ao longo do tempo. Ele ajusta uma função matemática a cada pixel, permitindo estimar amplitude, fase, tendência e comportamento sazonal.***

***- Amplitude (magnitude): representa a intensidade da oscilação anual do NDVI.***

***-Fase (phase): representa o deslocamento temporal da oscilação.***



In [1]:
import ee
import math
from typing import Callable
import matplotlib.pyplot as plt
import pandas as pd

from repository_ee import (
    Landsat8,
    Periodo,
    GeometryUtils,
    initialize_earth_engine,
    areaEstudo
)
initialize_earth_engine(use_eerepr=True)

In [2]:
araguaina  = areaEstudo.area("Araguaina")
aoi = GeometryUtils.polygon(araguaina)

### #1 Load a colection

In [3]:

from typing import Any
ANO: dict[str, Any] = {
    "inicio": 2015,
    "fim": 2026,
}
periodo = Periodo.geral(inicio=ANO["inicio"], fim = ANO["fim"])
imageLandsat = Landsat8.getSimpleCollection(geometry=aoi,period=periodo)

### step 2

set ndvi
set harmoni cycles
set list of harmoic frequencies
set the sequencie of band names for harmonic terms
set cos
set sen

In [4]:
#ndvi
dependentVariable = 'NDVI'
#cycles harmonic
#teste em 1 ciclo somente
#caso sete 2 config: # significa que irá criar as bandas espaçadas para ciclos de 1 ano como tambem para 6 meses
harmonicCycles = 2 
HarmonicFrequencies = ee.List.sequence(1, harmonicCycles) # neste caso uma sequencia de 1 para 1

print(HarmonicFrequencies)

## sequencie
def constructBandNames(
        base : str,
        bandList : ee.List
):
    return ee.List(bandList).map(lambda i: ee.String(base).cat(ee.Number(i).int()))

cosNames = constructBandNames('cos_',HarmonicFrequencies)
senNames = constructBandNames('sen_',HarmonicFrequencies)


independentsVariabels = ee.List(['constant','t']).cat(cosNames).cat(senNames)
independentsVariabels

ee.List({
  "functionInvocationValue": {
    "functionName": "List.sequence",
    "arguments": {
      "end": {
        "constantValue": 2
      },
      "start": {
        "constantValue": 1
      }
    }
  }
})


In [5]:
## add a ndvi band 

def addNdvi(
        image : ee.Image
) -> ee.Image:
    ndvi = image.normalizedDifference(['B5', 'B4']).rename('NDVI')
    return image.addBands(ndvi)




def add_dependents(image: ee.Image) -> ee.Image:
    """Adiciona bandas de constante (intercepto) e tempo em radianos a uma imagem.

    Prepara a imagem para análises de séries temporais no Google Earth Engine
    (como regressões lineares ou modelos harmônicos), calculando o tempo
    decorrido desde 1988-01-01 em radianos (onde 1 ano = 2π radianos).

    ria uma imagem onde todos os pixels têm o valor exato de 1 (por padrão, 
    o GEE nomeia essa banda como 'constant'). Em regressões lineares 
    ($y = a + bx$), essa banda representa o intercepto (o valor base $a$).

    Args:
        image: Uma imagem do Google Earth Engine (ee.Image) com propriedade de data.

    Returns:
        A imagem original acrescida das bandas 'constant' (valor 1) e 't' (tempo).
    """
    # Calcula o tempo em anos fracionários desde a época Unix (1970-01-01)
    years = image.date().difference("2015-01-01", "year")

    # Converte os anos para radianos (ciclo anual = 2 * pi) e cria a banda 't'
    time_radians = ee.Image(years.multiply(2 * math.pi)).rename("t")

    # Cria a banda constante com valor 1 (para o intercepto da regressão)
    constant = ee.Image(1)

    return image.addBands(constant).addBands(time_radians.float())




def add_harmonics(
    freqs: list[float | int],
    cos_names: list[str] | None = None,
    sin_names: list[str] | None = None,
) -> Callable[[ee.Image], ee.Image]:
    """Cria uma função para adicionar termos harmônicos (seno e cosseno) a imagens.

    Retorna uma função compatível com `ImageCollection.map()` que calcula
    componentes sazonais para as frequências fornecidas, assumindo que a
    imagem já possui a banda de tempo em radianos ('t').

    Args:
        freqs: Lista de frequências harmônicas (ex: [1, 2] para ciclos anuais
            e semestrais).
        cos_names: Nomes para as bandas de cosseno. Se omitido, gera nomes
            automaticamente no formato 'cos_1', 'cos_2', etc.
        sin_names: Nomes para as bandas de seno. Se omitido, gera nomes
            automaticamente no formato 'sin_1', 'sin_2', etc.

    Returns:
        Uma função que recebe uma `ee.Image` e a retorna com as novas bandas
        de cosseno e seno anexadas.
    """
    if cos_names is None:
        cos_names = [f"cos_{f}" for f in freqs]
    if sin_names is None:
        sin_names = [f"sin_{f}" for f in freqs]

    def _map_function(image: ee.Image) -> ee.Image:
        # Cria uma imagem multibanda com as frequências constantes
        frequencies = ee.Image.constant(freqs)

        # Seleciona a banda de tempo em radianos ('t')
        time = ee.Image(image).select("t")

        # Calcula os termos de cosseno e seno multiplicando tempo pela frequência
        cosines = time.multiply(frequencies).cos().rename(cos_names)
        sines = time.multiply(frequencies).sin().rename(sin_names)

        return image.addBands(cosines).addBands(sines)

    return _map_function




In [6]:
## opera os resultados na imagem
# Opção 1: Encadeando tudo de uma vez (mais limpo e recomendado)
harmonic_landsat = (
    imageLandsat
    .filterDate("2015-01-01", "2026-01-01")
    .map(addNdvi)
    .map(add_dependents)
    .map(add_harmonics(freqs=HarmonicFrequencies, cos_names=cosNames, sin_names=senNames))
    
)
print(imageLandsat.first().bandNames().getInfo())

['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'B10', 'B11', 'QA_PIXEL', 'QA_RADSAT', 'SAA', 'SZA', 'VAA', 'VZA']


In [7]:
# output regression reduction
#O que faz: Calcula uma regressão linear múltipla pelo método dos mínimos quadrados para cada pixel ao longo de toda a série temporal.
#Resultado: Uma única imagem onde cada pixel guarda uma matriz (array) com os coeficientes da equação ($b_0, b_1, b_2, \dots$) e os resíduos do modelo.
# Se independentsVariabels for uma ee.List (como no código original em JS):
harmonicTrend = (
    imageLandsat
    .select(independentsVariabels.add(dependentVariable))
    .reduce(ee.Reducer.linearRegression(independentsVariabels.length(), 1))
)


#O que faz: Desempacota o resultado matricial do Passo 1 em bandas normais do Earth Engine, fáceis de visualizar e usar em contas.
#.select('coefficients'): Isola apenas os coeficientes da regressão (descartando a banda de resíduos). Como há 1 variável dependente, o formato por pixel é uma matriz 2D de tamanho $N \times 1$ (ex: 4x1).
#.arrayProject([0]): Achata essa matriz de 2D para um vetor 1D de tamanho $N$, eliminando a dimensão extra.
#.arrayFlatten([independents]): Converte cada posição do vetor em uma banda separada e dá a elas os nomes exatos da lista independents (ex: bandas chamadas 'constant', 't', 'cos_1', 'sin_1').
# 3. Extrai os coeficientes
harmonicTrendCoefficients = (
    harmonicTrend
    .select("coefficients")
    .arrayProject([0])
    .arrayFlatten([independentsVariabels])
)
harmonicTrendCoefficients
#O que faz: Aplica a equação encontrada de volta em cada data da coleção para gerar a série temporal modelada (sem ruídos/nuvens).
#Para cada imagem da coleção, o código aplica a equação linear $\hat{y} = b_0 \cdot 1 + b_1 \cdot t + b_2 \cdot \cos(\omega t) + b_3 \cdot \sin(\omega t)$:
#.select(independents).multiply(harmonicTrendCoefficients): Multiplica cada banda independente daquela data pelo seu respectivo coeficiente encontrado no Passo 2.
#.reduce('sum'): Soma todos esses produtos em cada pixel para obter o valor final previsto ($\hat{y}$).
# .rename('fitted') e .addBands(...): Nomeia essa previsão como 'fitted' e a anexa à imagem original, permitindo comparar o dado real (dependent) com a curva ajustada (fitted) em um gráfico.
fittedHarmonic = imageLandsat.map(
    lambda image: image.addBands(
        image.select(independentsVariabels)
        .multiply(harmonicTrendCoefficients)
        .reduce(ee.Reducer.sum())
        .rename("fitted")
    )
)

In [8]:
# 1. Recarrega/prepara a coleção salvando o resultado na variável 'harmonic_landsat'
harmonic_landsat = (
    imageLandsat
    .filterDate("2015-01-01", "2026-01-01")
    .map(addNdvi)
    .map(add_dependents)
    .map(add_harmonics(freqs=HarmonicFrequencies, cos_names=cosNames, sin_names=senNames))
)

# 2. Calcula a regressão usando 'harmonic_landsat'
harmonicTrend = (
    harmonic_landsat
    .select(independentsVariabels.add(dependentVariable))
    .reduce(ee.Reducer.linearRegression(independentsVariabels.length(), 1))
)

# 3. Extrai os coeficientes
harmonicTrendCoefficients = (
    harmonicTrend
    .select("coefficients")
    .arrayProject([0])
    .arrayFlatten([independentsVariabels])
)

# 4. Calcula os valores ajustados ('fitted') usando 'harmonic_landsat'
fittedHarmonic = harmonic_landsat.map(
    lambda image: image.addBands(
        image.select(independentsVariabels)
        .multiply(harmonicTrendCoefficients)
        .reduce(ee.Reducer.sum())
        .rename("fitted")
    )
)

In [9]:
import matplotlib.pyplot as plt
import pandas as pd


def plot_harmonic_series(
    collection: ee.ImageCollection,
    roi: ee.Geometry,
    scale: int = 30,
) -> pd.DataFrame:
    """Extrai a série temporal média na ROI e plota os valores reais vs. ajustados.

    Substitui o `ui.Chart.image.series` do Code Editor do Earth Engine.

    Args:
        collection: Coleção contendo as bandas 'NDVI' e 'fitted'.
        roi: Geometria da região de interesse (ee.Geometry).
        scale: Resolução espacial em metros para a redução média.

    Returns:
        DataFrame do pandas com as datas e valores médios extraídos.
    """
    def _extract_mean(image: ee.Image) -> ee.Feature:
        stats = image.select(["fitted", "NDVI"]).reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=roi,
            scale=scale,
            maxPixels=1e9,
        )
        return ee.Feature(None, stats).set("date", image.date().format("YYYY-MM-dd"))

    # Extrai os dados do servidor do GEE para um DataFrame local
    features = collection.map(_extract_mean).getInfo()["features"]
    records = [f["properties"] for f in features]
    df = pd.DataFrame(records).dropna()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date")

    # Plota o gráfico comparando original (pontos) e ajustado (linha)
    plt.figure(figsize=(10, 4))
    plt.scatter(df["date"], df["NDVI"], label="NDVI Original", s=15, alpha=0.7)
    plt.plot(df["date"], df["fitted"], label="Modelo Harmônico (Fitted)", color="red", linewidth=1.2)
    plt.title("Harmonic model: original and fitted values")
    plt.xlabel("Data")
    plt.ylabel("NDVI")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()

    return df


import math
import ee


def compute_seasonality_rgb(
    coefficients: ee.Image,
    collection: ee.ImageCollection,
) -> ee.Image:
    sen_1 = coefficients.select("sen_1")
    cos_1 = coefficients.select("cos_1")
    sen_2 = coefficients.select("sen_2")
    cos_2 = coefficients.select("cos_2")
    t     = coefficients.select("t")

    # Magnitude combinada (norma L2 dos coeficientes do 1º e 2º ciclos)
    magnitude = (
        cos_1.pow(2)
        .add(sen_1.pow(2))
        .add(cos_2.pow(2))
        .add(sen_2.pow(2))
        .sqrt()
        .multiply(5)
    )

    # Fase do ciclo anual principal no intervalo [0, 1]
    phase = sen_1.atan2(cos_1).unitScale(-math.pi, math.pi)

    # Brilho baseado no NDVI médio
    val = collection.select("NDVI").reduce(ee.Reducer.mean()).multiply(1.5)

    # Converte espaço de cores HSV para RGB
    rgb = ee.Image.cat([phase, magnitude, val]).hsvToRgb()

    # Retorna as bandas RGB + as bandas originais nomeadas
    return rgb.addBands([
        phase.rename("phase"),
        magnitude.rename("magnitude"),
        val.rename("val"),
    ])

In [10]:
# --- Execução ---
import geemap
# 1. Plotar o gráfico da série temporal
#df_serie = plot_harmonic_series(fittedHarmonic, aoi, scale=30)

# 2. Gerar imagem RGB de sazonalidade e visualizar no mapa interativo (geemap)
seasonality = compute_seasonality_rgb(coefficients=harmonicTrendCoefficients,collection= fittedHarmonic)

#m = geemap.Map()
#m.centerObject(aoi, 11)
#m.addLayer(seasonality, {}, "Seasonality")
#m.addLayer(aoi, {}, "ROI")
#m  # Exibe o mapa em ambientes Jupyter/Colab

# VAl
-- No passo anterior do tutorial do Google Earth Engine, o val foi calculado apenas para preencher o terceiro canal (Value/Brilho) da composição colorida HSV $\rightarrow$ RGB (ee.Image.cat(phase, magnitude, val).hsvToRgb()), garantindo que áreas com baixo NDVI médio (como água ou solo exposto) aparecessem escuras no mapa, enquanto áreas densamente vegetadas aparecessem brilhantes. Para classificação matemática, o constant já cumpre esse papel.

# nota
-- Nota extra sobre redundância nesse mesmo bloco: Note que magnitude e phase também são derivados matematicamente de sin_1 e cos_1 (são apenas a conversão de coordenadas cartesianas para polares). Eles costumam ser mantidos juntos em modelos baseados em árvores (como Random Forest) porque facilitam cortes ortogonais no espaço de atributos — por exemplo, separar culturas agrícolas pela data exata do pico (phase) é muito mais fácil para uma árvore de decisão usando phase diretamente do que combinando sin_1 e cos_1.

In [17]:

from repository_ee import CollectionUtils
ASSET_LANDSAT_SAZONALITY = (
   "projects/lateral-spirit-514212-j0/assets/landsat8_sazonalidade_ndvi_v2_2015_epoch_araguaina"
)


landsat_SAZONALITY = CollectionUtils.image(ASSET_LANDSAT_SAZONALITY)


In [18]:
# preditores

preditores = ee.Image.cat([
    harmonicTrendCoefficients.select("constant"),
    landsat_SAZONALITY.select("magnitude").rename("magnitude"),
    landsat_SAZONALITY.select("phase").rename("phase"),
    harmonicTrendCoefficients.select("sen_1"),
    harmonicTrendCoefficients.select("cos_1"),
]).toFloat()
preditores



# tasks

In [19]:
from repository_ee import Exports

asset_root = "projects/lateral-spirit-514212-j0/assets"
export_region = aoi.bounds()
'''
task_preditores = Exports.image_to_assets(
    image=preditores,
    asset_id=f"{asset_root}/landsat8_preditores_harmonicos",
    description="landsat8_preditores_harmonicos",
    region=export_region,
    scale=30,
)
'''

Export_preditores_to_drive = Exports.image_to_drive(
    image=preditores,
    description="preditores_araguaina_v1",
    region=aoi
)

'''
task_seasonality = Exports.image_to_assets(
    image=seasonality,
    asset_id=f"{asset_root}/landsat8_sazonalidade_ndvi_v2_2015_epoch_araguaina",
    description="landsat8_sazonalidade_ndvi_v2_2015_epoch_araguaina",
    region=export_region,
    scale=30,
)
'''



Export_preditores_to_drive.start()
#task_seasonality.start()